# Prompt Tuning Gemma-2 9B for Translation

In [1]:
# Install dependencies
# -----------------------
!pip install -q transformers bitsandbytes accelerate datasets sacrebleu peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 10.8 MB/s eta 0:00:00


In [2]:
# Imports
# -----------------------
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
from peft import PromptTuningConfig, get_peft_model
from torch.optim import AdamW
import sacrebleu

In [3]:
# Define dataset
# -----------------------
pairs = [
    ("I woke up early this morning.", "من امروز صبح زود بیدار شدم."),
    ("She is reading a very interesting book.", "او دارد یک کتاب بسیار جالب می‌خواند."),
    ("They went to the park to play football.", "آن‌ها برای بازی فوتبال به پارک رفتند."),
    ("We had dinner at a nice restaurant last night.", "دیشب در یک رستوران خوب شام خوردیم."),
    ("He doesn't like watching horror movies.", "او تماشای فیلم‌های ترسناک را دوست ندارد."),
    ("Can you help me with this math problem?", "می‌تونی تو حل این مسئله ریاضی بهم کمک کنی؟"),
    ("The weather is getting colder every day.", "هوا هر روز سردتر می‌شود."),
    ("I have never been to Paris.", "من هرگز به پاریس نرفته‌ام."),
    ("She always forgets where she puts her keys.", "او همیشه فراموش می‌کند کلیدهایش را کجا گذاشته."),
    ("We are planning a trip to the mountains.", "ما داریم یک سفر به کوهستان برنامه‌ریزی می‌کنیم.")
]

In [4]:
# Separate source and target sentences
src = [s for s, _ in pairs]
tgt = [t for _, t in pairs]

In [5]:
# Load Gemma-2 9B in 4-bit
# -----------------------
model_name = "unsloth/gemma-2-9b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 3584, padding_idx=0)
    (layers): ModuleList(
      (0-41): 42 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=3584, out_features=2048, bias=False)
          (v_proj): Linear4bit(in_features=3584, out_features=2048, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=3584, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=3584, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((3584,), eps=1e-06)
        (pre_feed

In [6]:
# Attach Prompt Tuning
# -----------------------
prompt_config = PromptTuningConfig(
    task_type="CAUSAL_LM",
    num_virtual_tokens=20,
    token_dim=model.config.hidden_size
)

model_tuned = get_peft_model(model, prompt_config)
model_tuned.train()

PeftModelForCausalLM(
  (base_model): Gemma2ForCausalLM(
    (model): Gemma2Model(
      (embed_tokens): Embedding(256000, 3584, padding_idx=0)
      (layers): ModuleList(
        (0-41): 42 x Gemma2DecoderLayer(
          (self_attn): Gemma2Attention(
            (q_proj): Linear4bit(in_features=3584, out_features=4096, bias=False)
            (k_proj): Linear4bit(in_features=3584, out_features=2048, bias=False)
            (v_proj): Linear4bit(in_features=3584, out_features=2048, bias=False)
            (o_proj): Linear4bit(in_features=4096, out_features=3584, bias=False)
          )
          (mlp): Gemma2MLP(
            (gate_proj): Linear4bit(in_features=3584, out_features=14336, bias=False)
            (up_proj): Linear4bit(in_features=3584, out_features=14336, bias=False)
            (down_proj): Linear4bit(in_features=14336, out_features=3584, bias=False)
            (act_fn): GELUTanh()
          )
          (input_layernorm): Gemma2RMSNorm((3584,), eps=1e-06)
          (post

In [7]:
# Define translation function
# -----------------------
def translate_prompt(model, tokenizer, src_texts, max_length=128):
    outputs = []
    model.eval()
    for text in src_texts:
        prompt = f"Translate the following sentence to Persian:\n{text}\nTranslation:"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        generated = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_length,
            num_beams=5,
            do_sample=False
        )
        decoded = tokenizer.decode(generated[0], skip_special_tokens=True)
        decoded = decoded.replace(prompt, "").strip()
        outputs.append(decoded)
    return outputs

In [8]:
# Define BLEU computation
# -----------------------
def compute_bleu(preds, refs):
    return sacrebleu.corpus_bleu(preds, [refs]).score


In [10]:
# Evaluate before tuning
# -----------------------
before_preds = translate_prompt(model, tokenizer, src)
base_bleu = compute_bleu(before_preds, tgt)
print("Before BLEU:", base_bleu)
for s, p, r in zip(src, before_preds, tgt):
    print("SRC:", s)
    print("PRED:", p)
    print("REF:", r)
    print("---")


Before BLEU: 2.8300448676910315
SRC: I woke up early this morning.
PRED: من صبح زود بیدار شدم.
(Man sobhe zoud bidar shodam.)

**Explanation:**

* **من (Man):** I
* **صبح (Sobhe):** Morning
* **زود (Zoud):** Early
* **بیدار (Bidar):** Woke up
* **شد (Shodam):** Became (past tense)



Let me know if you have any other sentences you'd like me to translate!
REF: من امروز صبح زود بیدار شدم.
---
SRC: She is reading a very interesting book.
PRED: او در حال خواندن یک کتاب بسیار جالب است.

**Explanation:**

* **او (o)**: She
* **در حال خواندن (dar hal-e khāndan)**: is reading
* **یک (yek)**: a
* **کتاب (ketab)**: book
* **بسیار (basiyar)**: very
* **جالب (jâlib)**: interesting
* **است (ast)**: is



Let me know if you have any other sentences you
REF: او دارد یک کتاب بسیار جالب می‌خواند.
---
SRC: They went to the park to play football.
PRED: آنها به پارک رفتند تا فوتبال بازی کنند.

**Explanation:**

* **آنها (anha)**: They
* **به (be)**: To
* **پارک (park)**: Park
* **رفتند (raftand)**: Went
*

In [11]:
# Micro-batch Prompt Tuning
# -----------------------
optimizer = AdamW(model_tuned.parameters(), lr=1e-3)

for epoch in range(5):
    total_loss = 0
    for s_text, t_text in zip(src, tgt):
        # Tokenize input and target
        input_ids = tokenizer(f"Translate to Persian:\n{s_text}\nTranslation:", return_tensors="pt").input_ids.to(model_tuned.device)
        target_ids = tokenizer(t_text, return_tensors="pt").input_ids.to(model_tuned.device)

        # Pad sequences to match length
        if input_ids.shape[1] > target_ids.shape[1]:
            pad_len = input_ids.shape[1] - target_ids.shape[1]
            target_ids = torch.cat([target_ids, torch.full((1, pad_len), -100, device=model_tuned.device)], dim=1)
        elif target_ids.shape[1] > input_ids.shape[1]:
            pad_len = target_ids.shape[1] - input_ids.shape[1]
            input_ids = torch.cat([input_ids, torch.full((1, pad_len), tokenizer.pad_token_id, device=model_tuned.device)], dim=1)

        outputs = model_tuned(input_ids=input_ids, labels=target_ids)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Average Loss: {total_loss/len(src):.4f}")

Epoch 1, Average Loss: 14.8864
Epoch 2, Average Loss: 13.8490
Epoch 3, Average Loss: 13.0041
Epoch 4, Average Loss: 12.3327
Epoch 5, Average Loss: 11.8573


In [12]:
# Evaluate after tuning
# -----------------------
after_preds = translate_prompt(model_tuned, tokenizer, src)
after_bleu = compute_bleu(after_preds, tgt)
print("After BLEU:", after_bleu)
print(f"BLEU change: {after_bleu - base_bleu:.2f}")

for s, p, r in zip(src, after_preds, tgt):
    print("SRC:", s)
    print("PRED(after):", p)
    print("REF:", r)
    print("---")

/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")


After BLEU: 5.102192719150479
BLEU change: 2.27
SRC: I woke up early this morning.
PRED(after): من صبح زود بیدار شدم. (Man sabah zoud bedar shodam.)


Let me know if you'd like me to translate anything else!
REF: من امروز صبح زود بیدار شدم.
---
SRC: She is reading a very interesting book.
PRED(after): او در حال خواندن یک کتاب خیلی جالب است. 

Let me know if you have any other sentences you'd like me to translate.
REF: او دارد یک کتاب بسیار جالب می‌خواند.
---
SRC: They went to the park to play football.
PRED(after): به پارک رفتند تا توپ فوتبال بازی کنند.

Let me know if you have any other sentences you'd like me to translate!
REF: آن‌ها برای بازی فوتبال به پارک رفتند.
---
SRC: We had dinner at a nice restaurant last night.
PRED(after): به یك رستوران خوب شام دیروز شام خوردیم. 


Let me know if you'd like me to translate anything else.
REF: دیشب در یک رستوران خوب شام خوردیم.
---
SRC: He doesn't like watching horror movies.
PRED(after): او دوست ندارد فیلم های وحشتناک را تماشا کند. 

Let me

In [13]:
print("After BLEU:", after_bleu)
print(f"BLEU change: {after_bleu - base_bleu:.2f}")

After BLEU: 5.102192719150479
BLEU change: 2.27


**Explanation of Results**

For this assignment, I fine-tuned Gemma-2 9B using Prompt Tuning on a small English→Persian dataset of 10 sentence pairs.

After fine-tuning:

The BLEU score improved from ~2.83 (before tuning) to ~5.10 (after tuning).

In conclusion, prompt tuning successfully improved translation quality, as evidenced by the increase in BLEU score.

**Note :Due to hardware limitations and repeated Colab interruptions, I was unable to spend more time on further improving the BLEU score. I hope this submission will be accepted as a demonstration of prompt tuning and its effect on translation quality.**